<div text-align="center">
  <img src="https://raw.githubusercontent.com/FarnoushRJ/MambaLRP/main/assets/MambaLRP_logo.jpeg" width="1000"/>
</div>


<div text-align="center"><h1>🐍 MambaLRP is here! 🎉</h1>

Clone the repository and install MambaLRP.

In [1]:
# Cell 1: Clone
!git clone https://github.com/AdamBosch/MambaLRP.git || echo "Already cloned"

fatal: destination path 'MambaLRP' already exists and is not an empty directory.
Already cloned


In [2]:
# Cell 2: Pin torch first
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0+cu118 \
    --index-url https://download.pytorch.org/whl/cu118 -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import subprocess
result = subprocess.run(['find', '/home/jovyan', '-name', 'torch', '-type', 'd'], 
                       capture_output=True, text=True)
print(result.stdout)

/home/jovyan/.local/lib/python3.10/site-packages/torch
/home/jovyan/.local/lib/python3.10/site-packages/torch/include/torch
/home/jovyan/.local/lib/python3.10/site-packages/torch/include/torch/csrc/api/include/torch



In [4]:
# Alternative Cell 3 - use pip's --find-links and pre-built wheels
!PYTHONPATH=/home/jovyan/.local/lib/python3.10/site-packages \
    pip install causal-conv1d==1.2.0.post2 --no-build-isolation --no-cache-dir -q
!PYTHONPATH=/home/jovyan/.local/lib/python3.10/site-packages \
    pip install mamba-ssm==1.2.0.post1 --no-build-isolation --no-cache-dir -q


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
# Cell 4: Write constraints file and install the rest
!echo "torch==2.1.0+cu118" > /tmp/constraints.txt
!pip install "numpy<2.0.0" "pyarrow<15.0.0" datasets==2.14.5 \
    transformers==4.40.1 captum==0.7.0 einops \
    -c /tmp/constraints.txt -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
# Cell 5: Install MambaLRP
!pip install ./MambaLRP --no-deps -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
import torch
print(torch.__version__)       # should be 2.1.0+cu118
import causal_conv1d
import mamba_ssm
print("All imports OK")

2.1.0+cu118
All imports OK


Import necessary packages.

In [8]:
from transformers import MambaConfig, MambaForCausalLM, AutoTokenizer
import sys

from mamba_lrp.model.mamba_huggingface import ModifiedMambaForCausalLM
from mamba_lrp.model.utils import *
from mamba_lrp.lrp.utils import relevance_propagation
from mamba_lrp.dataset.general_dataset import get_sst_dataset, get_medbios_dataset
import torch
import numpy as np

## Load model

Load model and tokenizer.

In [9]:
!pip install gdown

# Import gdown
import gdown

# Define the file ID and the destination file name
file_id = '1PVaAR7esC5DeeJzR5s5xfO8w_gEmw1lS'  # MedBios
destination = 'mamba_medbios_weights.pt'

# Construct the URL
url = f'https://drive.google.com/uc?id={file_id}'

# Download the file
gdown.download(url, destination, quiet=False)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


Downloading...
From (original): https://drive.google.com/uc?id=1PVaAR7esC5DeeJzR5s5xfO8w_gEmw1lS
From (redirected): https://drive.google.com/uc?id=1PVaAR7esC5DeeJzR5s5xfO8w_gEmw1lS&confirm=t&uuid=fb6b044f-016a-480d-9f30-204baa09ef3f
To: /home/jovyan/mamba_medbios_weights.pt
100%|██████████| 517M/517M [00:05<00:00, 87.9MB/s] 


'mamba_medbios_weights.pt'

In [10]:
# Initialize tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
tokenizer.eos_token = "<|endoftext|>"
tokenizer.bos_token = "<|startoftext|>"
tokenizer.pad_token = "<|pad|>"
tokenizer.unk_token = "<|unkown|>"
tokenizer.add_tokens(['<|unkown|>', '<|pad|>', "<|startoftext|>"], special_tokens=True)

model = MambaForCausalLM.from_pretrained("state-spaces/mamba-130m-hf", use_cache=True)
resize_token_embeddings(model, len(tokenizer))
model.lm_head = torch.nn.Linear(768, 28, bias=True)

/home/jovyan/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [11]:
model.load_state_dict(
    torch.load('mamba_medbios_weights.pt', map_location=torch.device('cpu')),
    strict=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# Make model explainable.
modified_model = ModifiedMambaForCausalLM(model, is_fast_forward_available=False)
modified_model.eval()
model.backbone.embeddings.requires_grad = False
pretrained_embeddings = model.backbone.embeddings

## Load dataset

Load SST-2 dataset.

In [12]:
validation_dataset = get_medbios_dataset(
    tokenizer=tokenizer,
    truncation=False,
    max_length=None
    )

## Generate explanation

Generate explanation for one sample.

In [13]:
i = 413
input_ids = validation_dataset.__getitem__(i)['input_ids'].unsqueeze(0).to(device)
label = torch.tensor(validation_dataset.__getitem__(i)['label']).long().to(device)
idx = torch.where(input_ids == 0)[1] + 1
input_ids = input_ids[:, :idx]
embeddings = pretrained_embeddings(input_ids)

R, prediction = relevance_propagation(
    model=modified_model,
    embeddings=embeddings,
    targets=label,
    n_classes=28
    )

## Visualization

For simplicity, we use the visualization utilities in Captum to display the results.

In [14]:
from captum.attr import visualization as viz

In [15]:
tokens = []
for id in input_ids[0][1: -2]:
    tokens.append(tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens([id.item()])))
attributions = R[0][1: -2]
attributions = attributions / attributions.max()

In [16]:
# Visualize the attributions
viz.visualize_text([viz.VisualizationDataRecord(
    attributions,
    torch.max(model(input_ids).logits[:, -1, :], dim=1).values.item(),
    torch.argmax(model(input_ids).logits[:, -1, :], dim=1).item(),
    true_class=label.item(),
    attr_class=label.item(),
    attr_score=attributions.sum(),
    raw_input_ids=tokens,
    convergence_score=None
)])

In [17]:
if isinstance(attributions, np.ndarray):
    attributions_tensor = torch.from_numpy(attributions)
    print("here")
else:
    attributions_tensor = attributions
    

sorted_indices = torch.argsort(attributions_tensor, descending=True)

here


In [18]:
def get_morf_curve(model, input_ids, sorted_indices, target_label):
    scores = []
    temp_input_ids = input_ids.clone()
    
    # 0. Initial probability
    with torch.no_grad():
        # The modified_model returns the logits tensor directly
        logits = model(temp_input_ids) 
        # Select the last token's logits
        initial_logits = logits[:, -1, :]
        initial_prob = torch.softmax(initial_logits, dim=-1)[0, target_label].item()
        scores.append(initial_prob)

    # 1. Iteratively mask tokens and record probability
    for idx in sorted_indices:
        # idx is from attributions which excluded the first/last tokens
        # We add 1 to map back to the correct position in input_ids
        temp_input_ids[0, idx + 1] = tokenizer.pad_token_id 
        
        with torch.no_grad():
            logits = model(temp_input_ids)
            last_token_logits = logits[:, -1, :]
            prob = torch.softmax(last_token_logits, dim=-1)[0, target_label].item()
            scores.append(prob)
            
    return scores

# Now call it with the modified_model
morf_values = get_morf_curve(modified_model, input_ids, sorted_indices, label.item())

In [19]:
def get_morf_curve_logits(model, input_ids, sorted_indices, target_label):
    scores = []
    temp_input_ids = input_ids.clone()
    
    # 0. Initial Logit
    with torch.no_grad():
        logits = model(temp_input_ids) 
        # Select the target class logit for the last token
        initial_logit = logits[0, -1, target_label].item()
        scores.append(initial_logit)

    # 1. Iteratively mask tokens and record the logit
    for idx in sorted_indices:
        # Map back to the correct position in input_ids
        temp_input_ids[0, idx + 1] = tokenizer.pad_token_id 
        
        with torch.no_grad():
            logits = model(temp_input_ids)
            logit = logits[0, -1, target_label].item()
            scores.append(logit)
            
    return scores
morf_values = get_morf_curve_logits(modified_model, input_ids, sorted_indices, label.item())

In [23]:
def get_morf_curve_batched(model, input_ids, sorted_indices, target_label, chunk_size=8):
    seq_len = len(sorted_indices)
    # 1. Prepare the full sequence of masked inputs on CPU first
    batch_input = input_ids.repeat(seq_len + 1, 1).cpu() 
    
    for i, idx in enumerate(sorted_indices):
        batch_input[i+1:, idx + 1] = tokenizer.pad_token_id
        
    all_target_probs = []
    
    # 2. Process in smaller chunks to save VRAM
    with torch.no_grad():
        for i in range(0, batch_input.size(0), chunk_size):
            chunk = batch_input[i : i + chunk_size].to(device)
            logits = model(chunk)
            # Calculate probabilities for the target class
            logits = model(chunk)
            target_logits = logits[:, -1, target_label].cpu().numpy()
            all_target_probs.extend(target_logits.tolist())
            
            # 3. Explicitly clear the chunk from VRAM
            del logits, chunk
            # torch.cuda.empty_cache() # Optional: only if fragmentation is severe
            
    return all_target_probs

In [24]:
def get_delta_af_fast(model, input_ids, attributions, target_label):
    # 1. Prepare indices and batch
    # attributions exclude first/last tokens, so seq_len is len(attr)
    seq_len = len(attributions)
    idx_morf = torch.argsort(attributions, descending=True)
    idx_lerf = torch.argsort(attributions, descending=False)
    
    # We create two batches: one for MoRF, one for LeRF
    # Each batch size is (seq_len + 1)
    def compute_auc(indices):
        batch_input = input_ids.repeat(seq_len + 1, 1)
        for i, idx in enumerate(indices):
            # Apply cumulative masks
            # +1 because attributions start after the BOS token
            batch_input[i+1:, idx + 1] = tokenizer.pad_token_id
            
        with torch.no_grad():
            # Run the entire mask sequence through the GPU at once
            logits = model(batch_input)
            # Take the logit for the target class at the last position
            curve = logits[:, -1, target_label].cpu().numpy()
        return np.trapz(curve) / len(curve)

    af_morf = compute_auc(idx_morf)
    af_lerf = compute_auc(idx_lerf)
    
    return af_lerf - af_morf

In [25]:
import numpy as np
from tqdm import tqdm

num_samples = 50
all_delta_afs = []

print(f"Evaluating {num_samples} samples from Medical-Bios...")

for i in tqdm(range(num_samples)):
    # 1. Get sample
    sample = validation_dataset.__getitem__(i)
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    label = torch.tensor(sample['label']).long().to(device)
    
    # 2. Trim padding for LRP
    idx = torch.where(input_ids == 0)[1] + 1
    input_ids = input_ids[:, :idx]
    embeddings = pretrained_embeddings(input_ids)

    # 3. Generate LRP Attribution
    R, _ = relevance_propagation(
        model=modified_model,
        embeddings=embeddings,
        targets=label,
        n_classes=28
    )
    
    # Extract relevance for tokens (excluding BOS/EOS as in Cell 14)
    # Note: R shape is [batch, seq_len], we take [0, 1:-2]
    attr = R[0][1:-2]
    if isinstance(attr, np.ndarray):
        attr = torch.from_numpy(attr)
    
    # 4. Calculate MoRF and LeRF
    # MoRF (Most Relevant First)
    idx_morf = torch.argsort(attr, descending=True)
    morf_probs = get_morf_curve_batched(modified_model, input_ids, idx_morf, label.item())
    af_morf = np.trapz(morf_probs) / len(morf_probs)
    
    # LeRF (Least Relevant First)
    idx_lerf = torch.argsort(attr, descending=False)
    lerf_probs = get_morf_curve_batched(modified_model, input_ids, idx_lerf, label.item())
    af_lerf = np.trapz(lerf_probs) / len(lerf_probs)
    
    all_delta_afs.append(af_lerf - af_morf)
    del R, embeddings, input_ids, attr
    torch.cuda.empty_cache()

# Final Result
mean_delta_af = np.mean(all_delta_afs)
print(f"\nMean Delta AF over {num_samples} samples: {mean_delta_af:.4f}")
print(f"Paper Reference (Mamba-130M Medical-Bios): 3.906")

Evaluating 50 samples from Medical-Bios...


100%|██████████| 50/50 [07:02<00:00,  8.45s/it]


Mean Delta AF over 50 samples: -0.0037
Paper Reference (Mamba-130M Medical-Bios): 3.906
